# Data Cleaning

### County

In [1063]:
# (county_id, county_name)
# I've manually extract counties name from dataset.
import pandas as pd
df = pd.read_csv("./uncleaned_datasets/county_uncleaned.csv")
county = df[['County']].dropna().copy()
# make sure there are no whitespaces leading and tailing.
fuel_type['fuel_type'] = fuel_type['fuel_type'].str.strip()
county.to_csv("./cleaned_data/county.csv", index=False)
county.head()

,County
0,Alameda
1,Alpine
2,Amador
3,Butte
4,Calaveras


In [1064]:
# set zip-county lookup table using the county table we just made.
# Load the cleaned county CSV
county_df = pd.read_csv("./cleaned_data/county.csv")
county_df.reset_index(inplace=True)
# assign county_id to county_name like in the schema.
county_lookup = {}
for i, row in county_df.iterrows():
    county_name = row['County']
    county_id = i + 1  # start IDs from 1
    county_lookup[county_name] = county_id

### ZIP

In [1066]:
# (zip_code, county_id)
df = pd.read_csv("./uncleaned_datasets/zip_code_database.csv")
# select rows of 'state' == 'CA', columns == ['zip', 'county']
ca_zip = df[df['state'] == 'CA'][['zip', 'county']].copy()

# remove the word "county" in county column, so "Los Angeles County" becomes "Los Angeles".
ca_zip['county'] = ca_zip['county'].str.replace(' County', '').str.strip()
# map county_id to respective county_name
ca_zip['county_id'] = ca_zip['county'].map(county_lookup)

# check if there are rows with NaN value, there are.
ca_zip[ca_zip['county_id'].isna()]

,zip,county,county_id
40242,95214,NaN,NaN


In [1067]:
# drop NaN value rows
cleaned_zip = ca_zip.dropna().copy()
# Convert float to int for export
cleaned_zip['county_id'] = cleaned_zip['county_id'].astype(int)
# match column names to table column
cleaned_zip = cleaned_zip[['zip', 'county_id']]
cleaned_zip.to_csv("./cleaned_data/zip.csv", index=False)
cleaned_zip.head()

,zip,county_id
38211,90001,19
38212,90002,19
38213,90003,19
38214,90004,19
38215,90005,19


### ReportYear and ModelYear range

In [1092]:
# (report_year), (model_year)
# these 2 are dimension tables containing a range of years
years_range = pd.DataFrame({'report_year': list(range(2011, 2025))})
years_range.to_csv("./cleaned_data/report_year.csv", index=False)
years_range.to_csv("./cleaned_data/model_year.csv", index=False)
years_range.head()

,report_year
0,2011
1,2012
2,2013
3,2014
4,2015


### FuelType

In [1071]:
# (fuel_type_id, fuel_type)
df = pd.read_csv("./uncleaned_datasets/vehicle_fuel_type_count_2024.csv")
# extract unique fuel types
unique_fuels = df['Fuel'].dropna().unique() # although it didn't have any NaN here, it is a good practice to do filter.
fuel_type = pd.DataFrame(unique_fuels, columns=['fuel_type'])
fuel_type['fuel_type'] = fuel_type['fuel_type'].str.strip()
fuel_type.to_csv("./cleaned_data/fuel_type.csv", index=False)
fuel_type

,fuel_type
0,Gasoline
1,Flex-Fuel
2,Hybrid Gasoline
3,Battery Electric
4,Plug-in Hybrid
5,Diesel and Diesel Hybrid
6,Natural Gas
7,Hydrogen Fuel Cell
8,Other


### VehicleRecord

In [1073]:
# (record_id, zip_code, model_year, vehicle_count, fuel_type_id)
### cleaning data
import pandas as pd
import glob
# get all vehicle data from different years
veh_df = pd.read_csv("./uncleaned_datasets/vehicle_fuel_type_count_2024.csv", low_memory=False)
veh_df.columns = veh_df.columns.str.upper()

### 1. extract columns I need
veh_df = veh_df[['ZIP CODE', 'FUEL','MODEL YEAR','VEHICLES']]
# normalize data type to ensure they align with database
veh_df['MODEL YEAR'] = veh_df['MODEL YEAR'].astype(str).str.strip()
veh_df['ZIP CODE'] = veh_df['ZIP CODE'].astype(str).str.strip()
veh_df['FUEL'] = veh_df['FUEL'].str.strip()

### 2. model_year column
# remove rows that has "<" in the model_year, I don't need bundle data for my analysis
veh_df = veh_df[~veh_df['MODEL YEAR'].astype(str).str.contains('<')].copy()
# filter rows that is out of model_year table
model_yr_lkup = pd.read_csv("./cleaned_data/model_year.csv", dtype={'model_year': str})
veh_df = veh_df[veh_df['MODEL YEAR'].isin(model_year_df['model_year'])].copy()
veh_df['MODEL YEAR'] =veh_df['MODEL YEAR'].astype(int) # match data type to schema

### 3. zip_code column
# there is a value in zip_code called "OOS", out of state, this need to be removed
# run the zip code with the lookup table to make sure all zip codes match table value
veh_zip = veh_df.copy()
zip_lkup = pd.read_csv("./cleaned_data/zip.csv", dtype={'zip': str})
# filter rows that is out of zip table
veh_zip = veh_zip[veh_zip['ZIP CODE'].isin(zip_lkup['zip'])].copy()
veh_zip['ZIP CODE'] = veh_zip['ZIP CODE'].astype(int) # match data type to schema

veh_df.head()

,ZIP CODE,FUEL,MODEL YEAR,VEHICLES
0,90022,Gasoline,2012,280
1,93263,Gasoline,2015,9
2,93703,Gasoline,2014,83
5,94550,Gasoline,2011,363
7,92882,Gasoline,2013,337


In [1074]:
### 4. convert fuel_type into fuel_type_id
veh_fuel = veh_zip.copy()
fuel_df = pd.read_csv("./cleaned_data/fuel_type.csv")
fuel_lkup = {name: idx + 1 for idx, name in enumerate(fuel_df['fuel_type'])}
veh_fuel['fuel_type_id'] = veh_fuel['FUEL'].map(fuel_lkup)
veh_fuel['fuel_type_id'] = veh_fuel['fuel_type_id'].astype(int) # match data type to schema
# drop the "fuel_type", keeping only "fuel_type_id"
veh_fuel = veh_fuel.drop(columns=['FUEL'])

### remove model_year==2025 vehicle, because there is only partial of 2025 vehicle registered in 2024
veh_record = veh_fuel[~(veh_fuel['MODEL YEAR']==2025)]

### saved cleaned data
veh_record.to_csv("./cleaned_data/vehicle.csv", index=False)
veh_record.head()

,ZIP CODE,MODEL YEAR,VEHICLES,fuel_type_id
0,90022,2012,280,1
1,93263,2015,9,1
2,93703,2014,83,1
5,94550,2011,363,1
7,92882,2013,337,1


### GasStationRecord

In [1076]:
# (county_id, report_year, gas_station_count, gasoline_sales, diesel_sales)
# I've manually delete un-related information with excel, I'm using pandas to melt from wide to long dataset
# 1. Gas station count
gas_station = pd.read_csv("./uncleaned_datasets/gas_station_uncleaned.csv").dropna(how='all').dropna(axis=1,how='all')
gas_station = gas_station.melt(id_vars=['County'], var_name='year', value_name='gas_station_count')

# 2. Gasoline sales
gas_sales = pd.read_csv("./uncleaned_datasets/gasoline_sales.csv").dropna(how='all').dropna(axis=1,how='all')
gas_sales= gas_sales.melt(id_vars=['County'], var_name='year', value_name='gasoline_sales')

# 3. diesel sales
diesel_sales = pd.read_csv("./uncleaned_datasets/diesel_sales.csv").dropna(how='all').dropna(axis=1,how='all')
diesel_sales = diesel_sales.melt(id_vars=['County'], var_name='year', value_name='diesel_sales')

# 4. join all 3 data, remove NaN rows
merge1 = pd.merge(gas_station, gas_sales, on=['County', 'year'], how='left')
gas_merged = pd.merge(merge1, diesel_sales, on=['County', 'year'], how='left')
gas_merged = gas_merged[gas_merged['diesel_sales'].notna()]

# 5. swap 'County' into county_id
gas_merged['county_id'] = gas_merged['County'].map(county_lookup)
gas_merged = gas_merged[['county_id','year','gas_station_count','gasoline_sales','diesel_sales']]

# 6. make sure all data type is correct
gas_merged['county_id'] = gas_merged['county_id'].astype(int)
gas_merged['year'] = gas_merged['year'].astype(int)
gas_merged['gas_station_count'] = gas_merged['gas_station_count'].str.replace(',', '').astype(int)
gas_merged['gasoline_sales'] = gas_merged['gasoline_sales'].str.replace(',', '').astype(float)
gas_merged['diesel_sales'] = gas_merged['diesel_sales'].astype(float)
gas_merged.to_csv("./cleaned_data/gas_station.csv", index=False)
gas_merged.head()

,county_id,year,gas_station_count,gasoline_sales,diesel_sales
0,1,2012,379,568.0,36.0
1,3,2012,28,14.0,2.0
2,4,2012,102,78.0,9.0
3,5,2012,27,12.0,2.0
4,6,2012,20,10.0,5.0


### ChargingStationRecord

In [1078]:
# (county_id, report_year, charging_station_count)
import pandas as pd
import glob
files = glob.glob("./uncleaned_datasets/EV_Chargers_*.csv") # read in bulk
dfs = []
for file in files:
    EV_df = pd.read_csv(file).dropna(how='all').dropna(axis=1,how='all')
    # stamp county_id to county
    EV_df['county_id'] = EV_df['County'].map(county_lookup)
    EV_df = EV_df.dropna().copy() # remove rows that aren't on county list, e.g. 'total', 'unknown'.
    # extract year from file name and stamp it
    report_year = file.split("_")[-1].split(".")[0]
    EV_df['report_year'] = report_year

    # extract the column I want
    EV_df = EV_df[['county_id','report_year','Total']]
    
    # normalize data type to ensure it align with database
    EV_df['county_id'] = EV_df['county_id'].astype(int)
    EV_df['report_year'] = EV_df['report_year'].astype(int)
    EV_df['Total'] = EV_df['Total'].astype(int)

    dfs.append(EV_df)
# merge all dataframes into 1 dataframe and export
charging_station = pd.concat(dfs, ignore_index=True)
charging_station.to_csv("./cleaned_data/charging_station.csv", index=False)
charging_station.head()

,county_id,report_year,Total
0,1,2020,3353
1,2,2020,15
2,3,2020,40
3,4,2020,76
4,5,2020,7
